# Check if all datasets are organized well

Consequential structure/naming of files & folders
-hopefully already from fmriprep output

In [ ]:
import os
import os.path as op
import pandas as pd

In [19]:
datapool_root_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk'
dataset = 'smile' #'numberline' # 
bids_folder = op.join(datapool_root_folder, f'ds-{dataset}')

subList = [int(d[4:]) for d in os.listdir(bids_folder) if d.startswith('sub-')]
sesList = [1,2]

In [20]:
print(*subList, sep=' ')

101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 302 303 305 306 307 308 310 311 312 314 315 316


In [18]:
# check 
import json

task = 'digitorder'
task_js = json.load(open(op.join(bids_folder,f'task-{task}_bold.json')))
task_js['RepetitionTime']

2.1

In [21]:
import re
pattern = re.compile(
    r"sub-(?P<sub>[0-9]+)_"
    r"ses-(?P<ses>[0-9]+)_"
    r"task-(?P<task>[a-zA-Z0-9]+).*\.gii$"
)

In [24]:
fmriprep_root = op.join(bids_folder, "derivatives", "fmriprep")

rows = []

for sub in subList[:-10]:  # Exclude last 5 subjects for testing
    sub_id = sub
    sub_dir = op.join(fmriprep_root, f"sub-{sub}")

    # Find sessions for this subject
    sessions = [d for d in os.listdir(sub_dir) if d.startswith("ses-")]
    for ses in sorted(sessions):
        ses_id = ses.split("-")[1]

        func_dir = op.join(sub_dir, ses, "func")
        if not op.isdir(func_dir):
            continue
        
        # Scan functional files
        for f in os.listdir(func_dir):
            if not f.endswith("space-fsaverage5_hemi-L_bold.func.gii"):
                continue

            m = pattern.match(f)
            if m:
                task = m.group("task")
                rows.append((sub_id, ses_id, task, f))

# Build DataFrame
df = pd.DataFrame(rows, columns=["subject", "session", "task", "filename"])
df = df.set_index(["subject", "session", "task"]).sort_index()


In [25]:
df

filename
subject session task                                                         
101     1       magjudge    sub-101_ses-1_task-magjudge_run-1_space-fsaver...
                magjudge    sub-101_ses-1_task-magjudge_run-2_space-fsaver...
                magjudge    sub-101_ses-1_task-magjudge_run-3_space-fsaver...
                placevalue  sub-101_ses-1_task-placevalue_run-1_space-fsav...
                rest        sub-101_ses-1_task-rest_run-1_space-fsaverage5...
...                                                                       ...
302     2       rest        sub-302_ses-2_task-rest_run-1_space-fsaverage5...
303     1       magjudge    sub-303_ses-1_task-magjudge_run-1_space-fsaver...
                magjudge    sub-303_ses-1_task-magjudge_run-2_space-fsaver...
                magjudge    sub-303_ses-1_task-magjudge_run-3_space-fsaver...
                placevalue  sub-303_ses-1_task-placevalue_run-1_space-fsav...

[223 rows x 1 columns]